## Import the relevant Python libraries

In [ ]:
import pandas as pd
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

## Loading the dataset

In [ ]:
df = pd.read_csv('insurance_claims_top_100.csv')
df.head()

## Here we are initializing OpenAI and Pinecone

In [ ]:
client = OpenAI(api_key = '')
pc = Pinecone(api_key='')

## Create a name for the Pinecone index

In [ ]:
index_name = "insurance-claims"

# Start coding here
# Use as many cells as you like
pc.create_index(index_name,dimension=1536,metric='cosine',spec=ServerlessSpec(cloud='aws',region='us-east-1'))

index = pc.Index(index_name)

## Creating an embedding for the text:

In [ ]:
def get_embedding(text, model='text-embedding-3-small'):
    text = text.replace('\n',' ')
    return client.embeddings.create(input=[text], model=model).data[0].embedding

## Inserting the Embeddings into the Pinecone Vector DB:

In [ ]:
for idx, row in df.iterrows():
    embedding = get_embedding(row['ClaimDescription'])
    index.upsert([(str(row['ClaimNumber']),embedding)])

## Finding Similar Claims to the Query Asked:

In [ ]:
def find_similar_claims(query, top_k=5):
    query_embedding = get_embedding(query)
    results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True)
    return results

In [ ]:
query = "Car accident with rear-end collision"
similar_claims = find_similar_claims(query)

print(similar_claims) # This is a dictionary

## Output

In [ ]:
{'matches': [{'id': 'WC8133442',
              'metadata': {},
              'score': 0.89387393,
              'values': []},
             {'id': 'WC3625998', 'score': 0.978313208, 'values': []},
             {'id': 'WC7425192',
              'metadata': {},
              'score': 1.10004616,
              'values': []},
             {'id': 'WC9445451', 'score': 1.13993788, 'values': []},
             {'id': 'WC5114658', 'score': 1.15511799, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

## Most Similar Answer:

In [ ]:
closest_claim_id = similar_claims['matches'][0]['id']
closest_claim_description = df[df['ClaimNumber'] == closest_claim_id]['ClaimDescription'].to_numpy()[0]
closest_claim_description